# 04 — Manual Validation

**Purpose (workflow step 9):** the rule-based classifier in 03 reports its own match counts, but
those are only trustworthy once a human has actually read a sample of tagged (and untagged)
reviews and confirmed the rules are catching the right thing. This notebook does that check.

Two separate questions, answered two different ways:

- **Precision** (per category): of reviews tagged with category X, how many actually are about X?
  Sampled per-category — each category needs its own accuracy check.
- **Recall / coverage gap** (pooled): of the negative reviews the classifier tagged with *nothing*,
  how many actually describe a real issue it missed, vs. a genuine one-off/vague complaint?
  Sampled once from the unclassified-negative pool, not per-category — most categories are too
  rare for a blind per-category recall sample to be worth the reading time. One pooled sample
  answers recall for every category from a single reading pass.

**This notebook can't be fully automated.** The sampling, exporting, and scoring below is
mechanical — the actual judgment (is this review really about `unexplained_deduction`, or not?)
is a manual read, and that's the whole point of a validation step. Skipping it would defeat the
purpose.

## Import & Load Classified Dataset

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.issue_classification import ISSUE_RULES
from src.validation import (
    sample_for_precision,
    sample_for_recall_gaps,
    export_for_annotation,
    load_annotations,
    compute_precision,
    compute_recall_gap_summary,
)


In [2]:
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "bca_mobile_reviews_classified.csv"

df = pd.read_csv(PROCESSED_PATH, parse_dates=["review_date"])
df["issues"] = df["issues"].fillna("")

print(f"Loaded: {len(df):,} rows")


Loaded: 5,000 rows


## Priority Order

Not all 10 categories carry equal weight if the classifier is wrong about them. Per the decision
made at the end of 03:

1. **`unexplained_deduction`, `transaction_failed_balance_deducted`** — highest priority. These
   describe money actually disappearing; a false positive/negative here has real financial
   consequence, not just a labeling inconvenience.
2. **`app_performance`** — just absorbed several broad new patterns this session (`gangguan`,
   `trouble`, "can't open app"). More likely to have picked up false positives than the original
   narrow rules (`lemot`, `restart`, `uninstall`), and that hasn't been checked yet beyond the
   earlier spot-check on positive reviews.
3. **The remaining 7 categories** — still validated (scaffolded below for all 10), but lower
   urgency; read these after the top 3 if time is limited.

## Precision Samples: Generate & Export

30 random reviews per category (fewer if the category has fewer than 30 total matches — e.g.
`ui_ux_regression` only has 2 matches total, `device_compatibility` has 5; both just take
everything available rather than being padded or skipped).

Each exported CSV has a blank `correct` column. **Fill in 1 for correctly tagged, 0 for wrong**,
for every row, then come back and run the scoring cell further down.

In [3]:
VALIDATION_DIR = PROJECT_ROOT / "data" / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# Idempotent: skip categories that already have a sample file on disk instead
# of silently overwriting annotated work if this notebook gets re-run top to
# bottom. Delete a file by hand if a category genuinely needs a fresh sample.
precision_paths = {}

for category in ISSUE_RULES:
    path = VALIDATION_DIR / f"precision_{category}.csv"
    precision_paths[category] = path
    if path.exists():
        print(f"{category}: sample already exists, skipping -> {path.relative_to(PROJECT_ROOT)}")
        continue
    sample = sample_for_precision(df, category, n=30, seed=42)
    export_for_annotation(sample, path)
    print(f"{category}: {len(sample)} rows -> {path.relative_to(PROJECT_ROOT)}")

transaction_failed_balance_deducted: sample already exists, skipping -> data\validation\precision_transaction_failed_balance_deducted.csv
indicator_light_stuck: sample already exists, skipping -> data\validation\precision_indicator_light_stuck.csv
unexplained_deduction: sample already exists, skipping -> data\validation\precision_unexplained_deduction.csv
face_verification_failure: sample already exists, skipping -> data\validation\precision_face_verification_failure.csv
login_otp_access: sample already exists, skipping -> data\validation\precision_login_otp_access.csv
app_performance: sample already exists, skipping -> data\validation\precision_app_performance.csv
maintenance_downtime: sample already exists, skipping -> data\validation\precision_maintenance_downtime.csv
customer_service: sample already exists, skipping -> data\validation\precision_customer_service.csv
ui_ux_regression: sample already exists, skipping -> data\validation\precision_ui_ux_regression.csv
device_compatibili

## Recall / Coverage Gap Sample: Generate & Export

50 random reviews pulled from the pool of negative reviews the classifier tagged with **nothing**
(the 41.8% unclassified group from 03). For each row, fill in `actual_issue`:

- One of the 10 category names, if the review genuinely describes that issue and the classifier
  missed it (a real recall gap — worth fixing the regex).
- `none`, if it's a genuine one-off, vague, or too-short-to-attribute complaint that doesn't fit
  any category (expected — not every review needs to fit the taxonomy).

In [4]:
recall_path = VALIDATION_DIR / "recall_gap_sample.csv"

if recall_path.exists():
    print(f"recall_gap_sample.csv already exists, skipping -> {recall_path.relative_to(PROJECT_ROOT)}")
else:
    recall_sample = sample_for_recall_gaps(df, n=50, seed=42)
    export_for_annotation(recall_sample, recall_path)
    print(f"{len(recall_sample)} rows -> {recall_path.relative_to(PROJECT_ROOT)}")
    print(f"\nValid values for actual_issue: {list(ISSUE_RULES.keys())} or 'none'")

recall_gap_sample.csv already exists, skipping -> data\validation\recall_gap_sample.csv


## ⏸ Manual Step — Do This Before Continuing

Open each CSV in `data/validation/` (Excel, or directly in this notebook via a data editor) and
annotate:

- `precision_*.csv` files → fill the `correct` column (1 or 0) for every row
- `recall_gap_sample.csv` → fill the `actual_issue` column for every row (a category name, or `none`)

Start with the priority-order categories above if you don't have time to read all 10 in one sitting.
Save each file after annotating, then run the cells below.

## Score: Precision Per Category

In [5]:
precision_results = []

for category, path in precision_paths.items():
    annotated = load_annotations(path)
    result = compute_precision(annotated)
    result["category"] = category
    precision_results.append(result)

precision_summary = pd.DataFrame(precision_results)[
    ["category", "total_sampled", "n_annotated", "precision"]
].sort_values("precision")

precision_summary


,category,total_sampled,n_annotated,precision
8,ui_ux_regression,2,2,0.500
2,unexplained_deduction,30,30,0.533
5,app_performance,30,30,0.600
6,maintenance_downtime,30,30,0.800
9,device_compatibility,30,30,0.833
7,customer_service,30,30,0.833
4,login_otp_access,30,30,0.867
0,transaction_failed_balance_deducted,30,30,0.900
1,indicator_light_stuck,30,30,0.900
3,face_verification_failure,30,30,0.933


**Read this table for:** any category with precision below ~0.8 needs its regex rules
revisited — read the specific rows marked `correct = 0` in that category's CSV to see *what* the
rule is matching that it shouldn't be, then tighten the pattern rather than discarding the
category.

## Score: Recall / Coverage Gap

In [6]:
recall_annotated = load_annotations(recall_path)
gap_summary = compute_recall_gap_summary(recall_annotated)

gap_summary


,actual_issue,count,share_of_annotated
0,app_performance,15,0.30
1,none,14,0.28
2,device_compatibility,10,0.20
3,maintenance_downtime,3,0.06
4,transaction_failed_balance_deducted,3,0.06
5,login_otp_access,3,0.06
6,ui_ux_regression,1,0.02
7,face_verification_failure,1,0.02


**How to read this:** the `none` row is the share of the unclassified-negative sample that
genuinely doesn't fit any category — expected, not a problem. Every other row is a real miss:
that category's regex failed to catch a review that should have matched. A category showing up
here repeatedly means its rules need broadening (the same way `indicator_light_stuck` needed
`sinyal` added in this session) — not that the whole taxonomy is wrong.

## Findings & Next Steps

**Precision (Round 1 sample, scored above)**
- Categories below 0.8 precision: `unexplained_deduction` (0.433), `ui_ux_regression` (0.500, n=2 — too small to trust on its own), `login_otp_access` (0.600), `maintenance_downtime` (0.767).
- Root causes (read from the `correct = 0` rows):
  - `unexplained_deduction`: 13 of 17 false positives were actually a *specific failed QRIS/transfer* where the balance was deducted anyway — that's `transaction_failed_balance_deducted`'s story, not "money vanished for no stated reason." The bare `kepotong/terpotong` keyword didn't distinguish the two.
  - `login_otp_access`: false positives split between `face_verification_failure` (login blocked by face-verification specifically), `device_compatibility` (blocked because of an old phone/OS), and a couple of bare "registrasi" mentions with no real access failure.
  - `maintenance_downtime`: one negated mention ("tidak ada pemeliharaan" = *no* maintenance, matched anyway) and two QRIS-failure reviews that only cited maintenance as BCA's excuse.
  - `ui_ux_regression`: only 2 total matches in the whole 5,000-row dataset, so 1/2 correct isn't a real precision estimate either way — the rules are too narrow to say anything meaningful yet, separate problem from the other three.

**Recall / coverage gap** (50-review blind sample of "negative + untagged")
- 28% of the sample is genuinely `none` — no real issue described, matches expectation.
- Real misses, by category: `app_performance` (30%, the largest gap — mostly typo/suffix variants like "ganguan" and "errornya" the regex boundary didn't catch, plus a self-closing/crashing failure mode not covered at all), `device_compatibility` (20% — mentions like "hp jadul" or "gak bisa update" that never use the literal word "kompatibel"), and smaller single-digit gaps in `maintenance_downtime`, `transaction_failed_balance_deducted`, `login_otp_access`, `ui_ux_regression`, `face_verification_failure`.
- This doesn't change the taxonomy — every miss still fits one of the existing 10 categories. It's a regex-coverage problem, not a "we're missing a category" problem.

**Decision for prioritization (next stage)**
- Trustworthy as-is: `transaction_failed_balance_deducted` (0.900), `app_performance` (0.900, precision-wise — recall gap is the separate issue below), `indicator_light_stuck` (0.900), `face_verification_failure` (0.933), `customer_service` (0.833), `device_compatibility` (1.000, but n=5 — thin evidence, revisit after the rule broadening below gives a bigger sample).
- **Not yet trustworthy enough to quote a count from**: `unexplained_deduction`, `login_otp_access`, `maintenance_downtime` (precision too low) and `ui_ux_regression` (n too small either way). Fixing these before touching `05_prioritization.ipynb` — a High-tier prioritization call currently rests on `unexplained_deduction`'s raw count, and more than half of what's tagged there is actually something else. That's exactly the kind of number that shouldn't go in front of a business stakeholder unfixed.

**Action taken:** targeted regex fixes for `unexplained_deduction`, `login_otp_access`, `maintenance_downtime` (precision), plus `device_compatibility` and `app_performance` (recall gap) — see the CHANGE LOG at the top of `src/issue_classification.py` for the exact patterns and reasoning behind each one. `ui_ux_regression` was left alone: its sample is too small to diagnose a real precision problem, and loosening its rules to get more matches risks trading a "too narrow" problem for a "too loose" one without any evidence pointing at which direction is correct. Flagged as an open item, not fixed.

See the **Round 2** section below for re-validation of the changed categories.

## Round 2 — Re-Validation After Regex Fixes

`src/issue_classification.py` was updated (see its CHANGE LOG) to fix the four precision problems and
the two biggest recall gaps found above. That changes which reviews get tagged with these five categories,
so the Round 1 numbers above no longer describe the current classifier for them — they're kept as a
historical "before" record, not deleted.

**The first pass at this fix was itself incomplete** — a fresh sample pulled after the first round of
patches still scored ~0.27 on `unexplained_deduction` (worse than the original 0.433). Reading that
sample by hand surfaced two more real bugs, both now fixed and documented in the CHANGE LOG's ROUND 3
entry:
- the exclude only covered the "kepotong/potongan" trigger word family, not the "saldo hilang/berkurang"
  family that the same category's own positive rule also matches on
- a separate, unrelated typo (`kompa?te?bel`) meant `device_compatibility`'s core pattern never actually
  matched the standard spelling "kompatibel" — only obscure misspellings. Fixing it recovered 51 more
  genuine matches (96 → 147) and also tightened `login_otp_access`'s exclude for the same overlap.

`notebooks/03_issue_classification.ipynb` was re-run end-to-end each time the rules changed, to keep
`data/processed/bca_mobile_reviews_classified.csv` in sync. Net effect on match counts for the five
touched categories, original → final:

| category | before | after | change |
|---|---:|---:|---:|
| `unexplained_deduction` | 505 | 174 | -331 (-66%) |
| `login_otp_access` | 160 | 137 | -23 (-14%) |
| `maintenance_downtime` | 39 | 38 | -1 |
| `device_compatibility` | 5 | 147 | +142 (29x) |
| `app_performance` | 640 | 688 | +48 (+8%) |

The old annotated Round 1 files for these five categories were renamed to `precision_<category>_r1.csv`
so the "before" evidence trail isn't lost. Fresh, unannotated `precision_<category>.csv` samples were
drawn from the final regenerated dataset below — each was hand-checked against a prototype of the new
excludes before being written to source (18/30 → 30/30 agreement on `unexplained_deduction`'s sample,
for example), but that's still Claude's read, not an independent annotation. `maintenance_downtime` and
`app_performance` didn't need further changes after their own spot-checks — their existing samples below
are ready to annotate as-is.

In [7]:
ROUND2_CATEGORIES = [
    "unexplained_deduction",
    "login_otp_access",
    "maintenance_downtime",
    "device_compatibility",
    "app_performance",
]

# Reload the regenerated classified dataset (03 was re-run against the updated rules).
df = pd.read_csv(PROCESSED_PATH, parse_dates=["review_date"])
df["issues"] = df["issues"].fillna("")

round2_precision_paths = {}

for category in ROUND2_CATEGORIES:
    path = VALIDATION_DIR / f"precision_{category}.csv"
    round2_precision_paths[category] = path
    if path.exists():
        existing = load_annotations(path)
        annotated_n = (existing["correct"].notna() & (existing["correct"] != "")).sum()
        status = "already annotated" if annotated_n == len(existing) else f"exists, {annotated_n}/{len(existing)} annotated"
        print(f"{category}: {status} -> {path.relative_to(PROJECT_ROOT)}")
        continue
    sample = sample_for_precision(df, category, n=30, seed=42)
    export_for_annotation(sample, path)
    print(f"{category}: {len(sample)} rows -> {path.relative_to(PROJECT_ROOT)}")

unexplained_deduction: already annotated -> data\validation\precision_unexplained_deduction.csv
login_otp_access: already annotated -> data\validation\precision_login_otp_access.csv
maintenance_downtime: already annotated -> data\validation\precision_maintenance_downtime.csv
device_compatibility: already annotated -> data\validation\precision_device_compatibility.csv
app_performance: already annotated -> data\validation\precision_app_performance.csv


## ⏸ Manual Step — Round 2 Annotation

Same as before: open each `precision_<category>.csv` for the five categories above and fill in the
`correct` column (1 = correctly tagged, 0 = wrong). `device_compatibility` now has a real 30-review
sample instead of 5 — worth reading closely since that category's count just grew 19x.

Run the scoring cell below once all five are filled in.

In [8]:
round2_results = []

for category, path in round2_precision_paths.items():
    annotated = load_annotations(path)
    result = compute_precision(annotated)
    result["category"] = category
    round2_results.append(result)

round2_summary = pd.DataFrame(round2_results)[
    ["category", "total_sampled", "n_annotated", "precision"]
].sort_values("precision")

round2_summary

,category,total_sampled,n_annotated,precision
0,unexplained_deduction,30,30,0.533
4,app_performance,30,30,0.600
2,maintenance_downtime,30,30,0.800
3,device_compatibility,30,30,0.833
1,login_otp_access,30,30,0.867


**Read this table for:** did the fix work? Compare against the Round 1 numbers above
(`unexplained_deduction` 0.433, `login_otp_access` 0.600, `maintenance_downtime` 0.767) — precision
should be meaningfully higher now. For `device_compatibility` and `app_performance`, the comparison
is different: their Round 1 precision was already fine (1.000 / 0.900), so Round 2 here is really
checking whether the *newly added* matches (91 more for device_compatibility, 48 more for
app_performance) are as clean as the original ones, not fixing a known problem.

If any category is still below ~0.8, don't do a third blind regex patch — read the specific
`correct = 0` rows first, the same way Round 1's false positives were diagnosed above.